# 第一种修改方法

In [35]:
import xml.etree.ElementTree as ET

# 文件路径
filtered_xml = "/rd1/user/lit/project/sORFs/analysis/20250523_human_ms_run/output/db_search_20250523/Trypsin/CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2_3_1_7020_d/interact-CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.pep.xml"         # 保留的扫描（已过滤）
unfiltered_xml = "/rd1/user/lit/project/sORFs/raw_data/MS/Guomics_SP_MSdata/CAD20250514licq_BSEP_DDA_60min_rename/output/db_search_20250523/Trypsin/CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2_3_1_7020_d_copy_20250703/interact-CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.pep.xml"       # 原始未过滤文件
output_xml = "2_modified.pep.xml"  # 输出文件名

# 1. 读取1.pep.xml，提取所有scan号
filtered_scans = set()
for event, elem in ET.iterparse(filtered_xml, events=("start",)):
    # 兼容命名空间
    if elem.tag.endswith("spectrum_query"):
        scan = elem.attrib.get("start_scan")
        if scan:
            filtered_scans.add(scan)
print(len(filtered_scans))

# 2. 处理2.pep.xml
tree = ET.parse(unfiltered_xml)
root = tree.getroot()

# 自动提取命名空间
namespace = ""
for elem in root.iter():
    if elem.tag.endswith("spectrum_query"):
        namespace = elem.tag.split("}")[0] + "}"
        break
for spectrum_query in root.iter(f"{namespace}spectrum_query"):
    scan = str(spectrum_query.attrib.get("start_scan"))
    if scan not in filtered_scans:
        for peptideprophet_result in spectrum_query.iter(f"{namespace}peptideprophet_result"):
            peptideprophet_result.set("probability", "0")
    else:
        for peptideprophet_result in spectrum_query.iter(f"{namespace}peptideprophet_result"):
            peptideprophet_result.set("probability", "1")
# 注册为默认命名空间，避免ns0前缀
if namespace:
    ET.register_namespace('', namespace[1:-1])  # 去掉大括号
# 3. 保存结果
tree.write(output_xml, encoding="utf-8", xml_declaration=True)
print(f"已保存修改后的文件为: {output_xml}")

56933
已保存修改后的文件为: 2_modified.pep.xml


# 第二种修改方法

In [36]:
import xml.etree.ElementTree as ET

# 文件路径
unfiltered_xml = "/rd1/user/lit/project/sORFs/raw_data/MS/Guomics_SP_MSdata/CAD20250514licq_BSEP_DDA_60min_rename/output/db_search_20250523/Trypsin/CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2_3_1_7020_d_copy_20250703/interact-CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.pep.xml"       # 原始未过滤文件
output_xml = "unfiltered_modified_2.pep.xml"  # 输出文件名
# 2. 处理2.pep.xml
tree = ET.parse(unfiltered_xml)
root = tree.getroot()

# 自动提取命名空间
namespace = ""
for elem in root.iter():
    if elem.tag.endswith("spectrum_query"):
        namespace = elem.tag.split("}")[0] + "}"
        break
# 1. 收集所有 spectrum_query 的 peptideprophet_result 及其 probability
all_results = []
for spectrum_query in root.iter(f"{namespace}spectrum_query"):
    for peptideprophet_result in spectrum_query.iter(f"{namespace}peptideprophet_result"):
        prob = float(peptideprophet_result.attrib.get("probability", 0))
        all_results.append((peptideprophet_result, prob))

# 2. 按 probability 排序（降序，概率高的排前面）
all_results.sort(key=lambda x: x[1], reverse=True)

n = len(all_results)
top_n = int(n * 0.2)
bottom_n = int(n * 0.2)

# 3. 分配新 probability
for i, (peptideprophet_result, _) in enumerate(all_results):
    if i < top_n:
        peptideprophet_result.set("probability", "1")
    elif i >= n - bottom_n:
        peptideprophet_result.set("probability", "0")
    else:
        peptideprophet_result.set("probability", "0.5")

# 注册为默认命名空间，避免ns0前缀
if namespace:
    ET.register_namespace('', namespace[1:-1])  # 去掉大括号
# 3. 保存结果
tree.write(output_xml, encoding="utf-8", xml_declaration=True)
print(f"已保存修改后的文件为: {output_xml}")

已保存修改后的文件为: unfiltered_modified_2.pep.xml
